
# Hashtag-only Association Mining with View Rate Bins

This notebook runs association-rule mining where:
- Antecedents are hashtags extracted from `caption_text` only.
- Consequent is one of `low_view_rate`, `avg_view_rate`, `high_view_rate`.
- Performance target is only `view_rate = views_count / followers_count`.

Outputs are saved to `reports/`:
- `hashtag_view_rate_binned_data.csv`
- `hashtag_view_rate_transactions.csv`
- `hashtag_view_rate_hyperparameter_summary.csv`
- `hashtag_view_rate_best_rules.csv`
- `hashtag_view_rate_all_rules.csv`


In [1]:

from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, Sequence

import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder


In [2]:

VIEW_RATE_BINS = ["low_view_rate", "avg_view_rate", "high_view_rate"]
HASHTAG_RE = re.compile(r"(?<!\w)#([^\s#]+)", flags=re.UNICODE)


def compute_view_rate(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["followers_count"] = pd.to_numeric(out.get("followers_count"), errors="coerce")
    out["views_count"] = pd.to_numeric(out.get("views_count"), errors="coerce")
    out = out[out["followers_count"].notna() & out["views_count"].notna()].copy()
    out = out[out["followers_count"] > 0].copy()
    out["view_rate"] = out["views_count"] / out["followers_count"]
    out = out[out["view_rate"].notna() & (out["view_rate"] >= 0)].copy()
    return out.reset_index(drop=True)


def clip_view_rate_outliers(df: pd.DataFrame, quantile: float = 0.95) -> tuple[pd.DataFrame, float]:
    out = df.copy()
    q95 = float(out["view_rate"].quantile(quantile))
    out = out[out["view_rate"] <= q95].copy()
    return out.reset_index(drop=True), q95


def make_view_rate_bins(df: pd.DataFrame, q: int = 3) -> pd.DataFrame:
    out = df.copy()
    try:
        out["view_rate_bin"] = pd.qcut(out["view_rate"], q=q, labels=VIEW_RATE_BINS)
    except ValueError:
        ranks = out["view_rate"].rank(method="first")
        out["view_rate_bin"] = pd.qcut(ranks, q=q, labels=VIEW_RATE_BINS)
    out["view_rate_bin"] = out["view_rate_bin"].astype(str)
    return out


def extract_hashtags(text: Any) -> list[str]:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    matches = HASHTAG_RE.findall(str(text))
    out = []
    seen = set()
    for m in matches:
        tag = m.strip().strip(".,!?;:'\"()[]{}<>").lower()
        if not tag:
            continue
        item = f"hashtag={tag}"
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def build_hashtag_transactions(df: pd.DataFrame) -> tuple[list[list[str]], pd.DataFrame]:
    transactions = []
    rows = []
    for i, row in df.iterrows():
        hashtags = extract_hashtags(row.get("caption_text"))
        if not hashtags:
            continue
        bin_item = str(row["view_rate_bin"])
        tx = hashtags + [bin_item]
        transactions.append(tx)
        rows.append(
            {
                "source_row": int(i),
                "view_rate": float(row["view_rate"]),
                "view_rate_bin": bin_item,
                "hashtags": "|".join(hashtags),
                "transaction": "|".join(tx),
                "hashtag_count": len(hashtags),
            }
        )
    return transactions, pd.DataFrame(rows)


def _set_to_str(s: frozenset[str]) -> str:
    return "{" + ", ".join(sorted(s)) + "}"


def mine_rules_for_config(
    transactions: Sequence[Sequence[str]],
    min_support: float,
    min_confidence: float,
    min_lift: float,
    max_antecedent_len: int,
) -> tuple[dict, pd.DataFrame]:
    if not transactions:
        summary = {
            "min_support": min_support,
            "min_confidence": min_confidence,
            "min_lift": min_lift,
            "max_antecedent_len": max_antecedent_len,
            "frequent_itemsets_count": 0,
            "valid_rules_before_filtering": 0,
            "final_rule_count": 0,
            "high_view_rate_rule_count": 0,
            "avg_lift": 0.0,
            "max_lift": 0.0,
            "avg_confidence": 0.0,
            "max_confidence": 0.0,
        }
        return summary, pd.DataFrame()

    te = TransactionEncoder()
    te_arr = te.fit(transactions).transform(transactions)
    ohe = pd.DataFrame(te_arr, columns=te.columns_)

    freq = fpgrowth(ohe, min_support=min_support, use_colnames=True, max_len=max_antecedent_len + 1)
    if freq.empty:
        summary = {
            "min_support": min_support,
            "min_confidence": min_confidence,
            "min_lift": min_lift,
            "max_antecedent_len": max_antecedent_len,
            "frequent_itemsets_count": 0,
            "valid_rules_before_filtering": 0,
            "final_rule_count": 0,
            "high_view_rate_rule_count": 0,
            "avg_lift": 0.0,
            "max_lift": 0.0,
            "avg_confidence": 0.0,
            "max_confidence": 0.0,
        }
        return summary, pd.DataFrame()

    rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)
    valid_before = int(len(rules))

    if rules.empty:
        summary = {
            "min_support": min_support,
            "min_confidence": min_confidence,
            "min_lift": min_lift,
            "max_antecedent_len": max_antecedent_len,
            "frequent_itemsets_count": int(len(freq)),
            "valid_rules_before_filtering": 0,
            "final_rule_count": 0,
            "high_view_rate_rule_count": 0,
            "avg_lift": 0.0,
            "max_lift": 0.0,
            "avg_confidence": 0.0,
            "max_confidence": 0.0,
        }
        return summary, pd.DataFrame()

    rules = rules.assign(
        antecedent_len=rules["antecedents"].apply(len),
        consequent_len=rules["consequents"].apply(len),
        consequent_item=rules["consequents"].apply(lambda s: next(iter(s)) if len(s) == 1 else None),
    )

    mask = (
        (rules["consequent_len"] == 1)
        & (rules["consequent_item"].isin(VIEW_RATE_BINS))
        & (rules["antecedents"].apply(lambda s: len(s) > 0 and all(str(x).startswith("hashtag=") for x in s)))
        & (rules["antecedents"].apply(lambda s: all(str(x) not in VIEW_RATE_BINS for x in s)))
        & (rules["antecedent_len"] <= max_antecedent_len)
        & (rules["lift"] >= min_lift)
    )

    final_rules = rules[mask].copy()
    if not final_rules.empty:
        final_rules = final_rules.sort_values(["lift", "confidence", "support"], ascending=[False, False, False]).reset_index(drop=True)
        final_rules["antecedents_str"] = final_rules["antecedents"].apply(_set_to_str)
        final_rules["consequent_str"] = final_rules["consequents"].apply(_set_to_str)
        final_rules["min_support"] = min_support
        final_rules["min_confidence"] = min_confidence
        final_rules["min_lift"] = min_lift
        final_rules["max_antecedent_len"] = max_antecedent_len

    summary = {
        "min_support": min_support,
        "min_confidence": min_confidence,
        "min_lift": min_lift,
        "max_antecedent_len": max_antecedent_len,
        "frequent_itemsets_count": int(len(freq)),
        "valid_rules_before_filtering": valid_before,
        "final_rule_count": int(len(final_rules)),
        "high_view_rate_rule_count": int((final_rules["consequent_item"] == "high_view_rate").sum()) if not final_rules.empty else 0,
        "avg_lift": float(final_rules["lift"].mean()) if not final_rules.empty else 0.0,
        "max_lift": float(final_rules["lift"].max()) if not final_rules.empty else 0.0,
        "avg_confidence": float(final_rules["confidence"].mean()) if not final_rules.empty else 0.0,
        "max_confidence": float(final_rules["confidence"].max()) if not final_rules.empty else 0.0,
    }

    return summary, final_rules


def run_hyperparameter_search(transactions: Sequence[Sequence[str]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    supports = [0.01, 0.02, 0.03, 0.05]
    confidences = [0.4, 0.5, 0.6]
    lifts = [1.0, 1.2, 1.5, 2.0]
    max_lens = [1, 2, 3]

    summaries = []
    all_rules = []

    for ms in supports:
        for mc in confidences:
            for ml in lifts:
                for max_len in max_lens:
                    summary, rules_df = mine_rules_for_config(transactions, ms, mc, ml, max_len)
                    summaries.append(summary)
                    if not rules_df.empty:
                        all_rules.append(rules_df)

    summary_df = pd.DataFrame(summaries).sort_values(
        ["high_view_rate_rule_count", "final_rule_count", "avg_lift", "avg_confidence"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    all_rules_df = pd.concat(all_rules, ignore_index=True) if all_rules else pd.DataFrame()
    return summary_df, all_rules_df


def choose_best_config(summary_df: pd.DataFrame, all_rules_df: pd.DataFrame) -> dict[str, Any]:
    eligible = summary_df[
        (summary_df["final_rule_count"] >= 10)
        & (summary_df["high_view_rate_rule_count"] >= 3)
        & (summary_df["min_lift"] >= 1.2)
    ].copy()

    if not eligible.empty:
        eligible = eligible.sort_values(
            ["high_view_rate_rule_count", "final_rule_count", "max_antecedent_len", "avg_lift", "avg_confidence"],
            ascending=[False, False, True, False, False],
        )
        best = eligible.iloc[0].to_dict()
    else:
        fallback = summary_df.sort_values(
            ["high_view_rate_rule_count", "avg_lift", "final_rule_count", "avg_confidence"],
            ascending=[False, False, False, False],
        )
        best = fallback.iloc[0].to_dict()

    if all_rules_df.empty:
        best_rules = pd.DataFrame()
    else:
        best_rules = all_rules_df[
            (all_rules_df["min_support"] == best["min_support"])
            & (all_rules_df["min_confidence"] == best["min_confidence"])
            & (all_rules_df["min_lift"] == best["min_lift"])
            & (all_rules_df["max_antecedent_len"] == best["max_antecedent_len"])
        ].copy()
        best_rules = best_rules.sort_values(["lift", "confidence", "support"], ascending=[False, False, False]).reset_index(drop=True)

    return {"best_config": best, "best_rules_df": best_rules}


def save_outputs(
    binned_df: pd.DataFrame,
    transactions_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    best_rules_df: pd.DataFrame,
    all_rules_df: pd.DataFrame,
    reports_dir: Path,
) -> dict[str, Path]:
    reports_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "binned": reports_dir / "hashtag_view_rate_binned_data.csv",
        "transactions": reports_dir / "hashtag_view_rate_transactions.csv",
        "summary": reports_dir / "hashtag_view_rate_hyperparameter_summary.csv",
        "best_rules": reports_dir / "hashtag_view_rate_best_rules.csv",
        "all_rules": reports_dir / "hashtag_view_rate_all_rules.csv",
    }
    binned_df.to_csv(paths["binned"], index=False)
    transactions_df.to_csv(paths["transactions"], index=False)
    summary_df.to_csv(paths["summary"], index=False)
    best_rules_df.to_csv(paths["best_rules"], index=False)
    all_rules_df.to_csv(paths["all_rules"], index=False)
    return paths


def run_hashtag_view_rate_pipeline(df: pd.DataFrame, reports_dir: Path):
    original_rows = len(df)
    vr_df = compute_view_rate(df)
    valid_rows = len(vr_df)

    clipped_df, q95 = clip_view_rate_outliers(vr_df, quantile=0.95)
    binned_df = make_view_rate_bins(clipped_df, q=3)

    transactions, transactions_df = build_hashtag_transactions(binned_df)
    summary_df, all_rules_df = run_hyperparameter_search(transactions)

    selected = choose_best_config(summary_df, all_rules_df)
    best_config = selected["best_config"]
    best_rules_df = selected["best_rules_df"]

    save_paths = save_outputs(
        binned_df=binned_df,
        transactions_df=transactions_df,
        summary_df=summary_df,
        best_rules_df=best_rules_df,
        all_rules_df=all_rules_df,
        reports_dir=reports_dir,
    )

    report = {
        "original_rows": original_rows,
        "valid_rows": valid_rows,
        "q95": q95,
        "rows_after_clipping": len(clipped_df),
        "num_transactions": len(transactions_df),
        "bin_counts": binned_df["view_rate_bin"].value_counts().reindex(VIEW_RATE_BINS, fill_value=0).to_dict(),
        "best_config": best_config,
        "save_paths": save_paths,
    }
    return report, summary_df, best_rules_df, all_rules_df, transactions_df, binned_df


In [3]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "data_processed.json"
REPORTS_DIR = PROJECT_ROOT / "reports"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {DATA_PATH}")

df = pd.read_json(DATA_PATH)
report, summary_df, best_rules_df, all_rules_df, transactions_df, binned_df = run_hashtag_view_rate_pipeline(df, REPORTS_DIR)

print("=== Final concise report ===")
print(f"Original row count: {report['original_rows']}")
print(f"Valid rows after view_rate calculation: {report['valid_rows']}")
print(f"q95 threshold: {report['q95']:.6f}")
print(f"Rows after clipping: {report['rows_after_clipping']}")
print(f"Number of transactions: {report['num_transactions']}")
print("Bin counts:", report["bin_counts"])

print("\nBest hyperparameters:")
print(report["best_config"])


=== Final concise report ===
Original row count: 710
Valid rows after view_rate calculation: 710
q95 threshold: 17.839547
Rows after clipping: 674
Number of transactions: 159
Bin counts: {'low_view_rate': 225, 'avg_view_rate': 225, 'high_view_rate': 224}

Best hyperparameters:
{'min_support': 0.01, 'min_confidence': 0.4, 'min_lift': 1.2, 'max_antecedent_len': 3.0, 'frequent_itemsets_count': 13772.0, 'valid_rules_before_filtering': 157529.0, 'final_rule_count': 3872.0, 'high_view_rate_rule_count': 1850.0, 'avg_lift': 1.5963106563318012, 'max_lift': 3.6136363636363638, 'avg_confidence': 0.5212110230826809, 'max_confidence': 1.0}


In [11]:
# Total number of transactions/posts/etc.
n = len(transactions_df)

top20 = best_rules_df[
    ["antecedents_str", "consequent_item", "support", "confidence", "lift"]
].copy()

top20["count"] = (top20["support"] * n).round().astype(int)

# Keep only rules with count > 10
top20 = top20[top20["count"] > 8].copy()

# Sort and take top 20 after filtering (optional but recommended)
top20 = top20.sort_values(
    ["lift", "confidence", "support"], ascending=[False, False, False]
).head(20)

top20 = top20[
    ["antecedents_str", "consequent_item", "count", "support", "confidence", "lift"]
]

top20


,antecedents_str,consequent_item,count,support,confidence,lift
14,{hashtag=متوفر_توصيل_لجميع_مناطق_الضفة_القدس_و...,low_view_rate,9,0.056604,1.000000,3.000000
157,{hashtag=فانيلا_اليوم_وكل_يوم},avg_view_rate,13,0.081761,0.764706,1.961101
2074,{hashtag=رام_الله},avg_view_rate,16,0.100629,0.615385,1.578164
2092,{hashtag=نابلس},avg_view_rate,9,0.056604,0.500000,1.282258


In [4]:

summary_df.head(15)


,min_support,min_confidence,min_lift,max_antecedent_len,frequent_itemsets_count,valid_rules_before_filtering,final_rule_count,high_view_rate_rule_count,avg_lift,max_lift,avg_confidence,max_confidence
0,0.01,0.4,1.0,3,13772,157529,3915,1850,1.591331,3.613636,0.520381,1.0
1,0.01,0.4,1.2,3,13772,157529,3872,1850,1.596311,3.613636,0.521211,1.0
2,0.01,0.4,1.5,3,13772,157529,2081,1850,1.865803,3.613636,0.539156,1.0
3,0.01,0.5,1.0,3,13772,151315,3852,1831,1.596440,3.613636,0.521624,1.0
4,0.01,0.5,1.2,3,13772,151315,3852,1831,1.596440,3.613636,0.521624,1.0
5,0.01,0.5,1.5,3,13772,151315,2062,1831,1.868338,3.613636,0.540068,1.0
6,0.02,0.4,1.0,3,12856,151389,3634,1794,1.554280,3.613636,0.504166,1.0
7,0.02,0.4,1.2,3,12856,151389,3592,1794,1.559078,3.613636,0.504845,1.0
8,0.02,0.4,1.5,3,12856,151389,1838,1794,1.822346,3.613636,0.509117,1.0
9,0.02,0.5,1.0,3,12856,145535,3577,1780,1.558994,3.613636,0.505091,1.0


In [6]:

top20 = best_rules_df[["antecedents_str", "consequent_item", "support", "confidence", "lift"]].head(20)
top20


,antecedents_str,consequent_item,support,confidence,lift
0,{hashtag=55555},high_view_rate,0.044025,1.0,3.613636
1,{hashtag=حملة_جراند_أكبر_وأوفر},high_view_rate,0.031447,1.0,3.613636
2,{hashtag=gym},high_view_rate,0.018868,1.0,3.613636
3,{hashtag=xtreme_fitness💪2025💪},high_view_rate,0.018868,1.0,3.613636
4,{hashtag=القدس},high_view_rate,0.012579,1.0,3.613636
5,"{hashtag=gym, hashtag=xtreme_fitness💪2025💪}",high_view_rate,0.012579,1.0,3.613636
6,{hashtag=fitnessgoal},high_view_rate,0.012579,1.0,3.613636
7,"{hashtag=fitnessgoal, hashtag=xtreme_fitness💪2...",high_view_rate,0.012579,1.0,3.613636
8,{hashtag=ramallah_city},high_view_rate,0.012579,1.0,3.613636
9,{hashtag=fitnessmodel},high_view_rate,0.012579,1.0,3.613636


In [7]:

print("Short interpretation:")
if best_rules_df.empty:
    print("No qualifying rules found under current constraints.")
else:
    focus = best_rules_df[best_rules_df["consequent_item"] == "high_view_rate"].head(5)
    if focus.empty:
        focus = best_rules_df.head(5)
    for _, r in focus.iterrows():
        line = (
            f"- {r['antecedents_str']} -> {r['consequent_item']} "
            f"(lift={r['lift']:.3f}, confidence={r['confidence']:.3f}, support={r['support']:.3f})"
        )
        print(line.encode('unicode_escape').decode('ascii'))


Short interpretation:
- {hashtag=55555} -> high_view_rate (lift=3.614, confidence=1.000, support=0.044)
- {hashtag=\u062d\u0645\u0644\u0629_\u062c\u0631\u0627\u0646\u062f_\u0623\u0643\u0628\u0631_\u0648\u0623\u0648\u0641\u0631} -> high_view_rate (lift=3.614, confidence=1.000, support=0.031)
- {hashtag=gym} -> high_view_rate (lift=3.614, confidence=1.000, support=0.019)
- {hashtag=xtreme_fitness\U0001f4aa2025\U0001f4aa} -> high_view_rate (lift=3.614, confidence=1.000, support=0.019)
- {hashtag=\u0627\u0644\u0642\u062f\u0633} -> high_view_rate (lift=3.614, confidence=1.000, support=0.013)


In [8]:

list(report["save_paths"].values())


[WindowsPath('c:/univ/Data mining/Project/reports/hashtag_view_rate_binned_data.csv'),
 WindowsPath('c:/univ/Data mining/Project/reports/hashtag_view_rate_transactions.csv'),
 WindowsPath('c:/univ/Data mining/Project/reports/hashtag_view_rate_hyperparameter_summary.csv'),
 WindowsPath('c:/univ/Data mining/Project/reports/hashtag_view_rate_best_rules.csv'),
 WindowsPath('c:/univ/Data mining/Project/reports/hashtag_view_rate_all_rules.csv')]